https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

In [1]:
import re
import string
from collections import Counter
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# one time download
nltk.download('stopwords', quiet = True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet = True)

True

In [2]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (5572, 2)
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [12]:
# Problem 2

# Find out the number of words in the entire corpus and also the total number of unique words(vocabulary) using just python
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess(text):
  text = text.lower()
  text = re.sub(r'[^a-z\s]', '', text)
  text = re.sub(r'\s+', ' ', text).strip()
  tokens = word_tokenize(text)
  tokens = [t for t in tokens if t not in stop_words]
  tokens = [lemmatizer.lemmatize(t) for t in tokens]
  return tokens

df['clean_tokens'] = df['message'].apply(preprocess)
df['clean_text'] = df['clean_tokens'].apply(lambda toks: ' '.join(toks))
print(df.columns.tolist())

['label', 'message', 'clean_tokens', 'clean_text']


In [13]:
# Problem 2

# Find out the number of words in the entire corpus and also the total number of unique words(vocabulary) using just python

all_words = []
for tokens in df['clean_tokens']:
    all_words.extend(tokens)

total_word_count = len(all_words)
unique_words = set(all_words)
vocab_size = len(unique_words)

print("Total number of words in corpus:", total_word_count)
print("Vocabulary size (unique words):", vocab_size)


Total number of words in corpus: 49528
Vocabulary size (unique words): 7877


In [14]:
# Problem 3

# Apply One Hot Encoding
vocab_list = sorted(unique_words)
word_to_index = {word: i for i, word in enumerate(vocab_list)}
encoder = OneHotEncoder()
word_array = [[w] for w in vocab_list]
onehot_matrix = encoder.fit_transform(word_array).toarray()

print("OneHot Encoding matrix shape:", onehot_matrix.shape)
print("Example one hot vector for the word '%s':"% vocab_list[0])
print(onehot_matrix[0])

OneHot Encoding matrix shape: (7877, 7877)
Example one hot vector for the word 'aa':
[1. 0. 0. ... 0. 0. 0.]


In [15]:
# Problem 4

# Apply bag words and find the vocabulary also find the times each word has occured
cv = CountVectorizer()
bow_matrix = cv.fit_transform(df['clean_text'])
bow_vocab = cv.get_feature_names_out()
word_freq = Counter(dict(zip(bow_vocab, bow_matrix.sum(axis=0).A1)))
print("Bow vocabulary size:", len(bow_vocab))
print("Top 15 most frequent words:")
for word, freq in word_freq.most_common(15):
  print(f" {word}: {freq}")

Bow vocabulary size: 7855
Top 15 most frequent words:
 call: 603
 im: 474
 get: 401
 ur: 384
 go: 308
 dont: 290
 free: 278
 ok: 277
 ltgt: 276
 know: 267
 day: 255
 got: 251
 come: 247
 like: 245
 ill: 241


In [16]:
# Problem 5

# Apply bag of bi-gram and bag of tri-gram and write down your observation about the dimensionality of the vocabulary
cv_bigram = CountVectorizer(ngram_range=(2,2))
bigram_matrix = cv_bigram.fit_transform(df['clean_text'])

cv_trigram = CountVectorizer(ngram_range=(3,3))
trigram_matrix = cv_trigram.fit_transform(df['clean_text'])

print("Unigram vocabulary size :", len(bow_vocab))
print("Bigram vocabulary size :", bigram_matrix.shape[1])
print("Trigram vocabulary size :", trigram_matrix.shape[1])

print("""
Observation:
As we move from unigram -> bigram -> trigram, the vocabulary size (dimensionality increases sharply. This happens because word combinations are far less repetitive than single words - most 2-word and 3-word sequences occur only once or twice in the corpus. So higher-order n-grams captur more context/phrase-level meaning, but the featrue space becomes much more sparse and high-dimensional, which needs more data and memory to model well.)
""")

Unigram vocabulary size : 7855
Bigram vocabulary size : 30085
Trigram vocabulary size : 30028

Observation:
As we move from unigram -> bigram -> trigram, the vocabulary size (dimensionality increases sharply. This happens because word combinations are far less repetitive than single words - most 2-word and 3-word sequences occur only once or twice in the corpus. So higher-order n-grams captur more context/phrase-level meaning, but the featrue space becomes much more sparse and high-dimensional, which needs more data and memory to model well.)



In [17]:
# Problem 6

# Apply tf-idf and find out the idf scores of words
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['clean_text'])
idf_scores = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))
idf_sorted = sorted(idf_scores.items(), key=lambda x: x[1])
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("\n10 words with the LOWEST idf (most common across messages):")
for word, score in idf_sorted[:10]:
  print(f" {word}: {score:.4f}")

print("\n10 words with the HIGHTS idf (rarest access messages):")
for word, score in idf_sorted[-10:]:
  print(f" {word}: {word}: {score:4f}")

TF-IDF matrix shape: (5572, 7855)

10 words with the LOWEST idf (most common across messages):
 call: 3.3140
 im: 3.5550
 get: 3.6882
 ur: 3.8988
 go: 3.9593
 ok: 4.0236
 dont: 4.0727
 know: 4.1042
 got: 4.1450
 day: 4.1534

10 words with the HIGHTS idf (rarest access messages):
 zealand: zealand: 8.932542
 zebra: zebra: 8.932542
 zero: zero: 8.932542
 zf: zf: 8.932542
 zhong: zhong: 8.932542
 zindgi: zindgi: 8.932542
 zogtorius: zogtorius: 8.932542
 zoom: zoom: 8.932542
 zouk: zouk: 8.932542
 zyada: zyada: 8.932542
